# RAHUL HIRUR

## Packages

Neptune used for tracking

In [1]:
!pip install neptune --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.9/487.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00


In [2]:
import networkx as nx # For graphs
import pickle # For data parsing
from networkx.algorithms.approximation import greedy_tsp # For approx TSP

In [3]:
import torch
import numpy as np
import transformers
from torch.utils.data import Dataset, DataLoader, random_split

import torch.nn as nn
import torch.nn.functional as F

import math
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import neptune

Mount the drive for data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Hyper parameter

In [5]:
#Initialize Neptune tracker
run = neptune.init_run(
    project="rahulhirur/DL4-Transformer",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIxMjkzM2M3My02ZWQ3LTQ4MjUtYTc2Zi03ODg3OWI4NDU3MGEifQ==",
)

[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/rahulhirur/DL4-Transformer/e/DLTRAN-28


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 32
dummy_data = True
epochs = 5
seed = 42
learning_rate = 1e-3

num_cities=20 # Initialized as 20 because all the data given is 20
d_model_enc=32
d_model_dec=32
d_model_ff=64
nhead=8
num_layers_enc=3
num_layers_dec=3
dropout_rate=0.3

run["parameters/device"] = device
run["parameters/batch_size"] = batch_size
run["parameters/dummy_data"] = dummy_data
run["parameters/epochs"] = epochs
run["parameters/seed"] = seed
run["parameters/learning_rate"] = learning_rate

run["parameters/num_cities"] = num_cities
run["parameters/d_model_enc"] = d_model_enc
run["parameters/d_model_dec"] = d_model_dec
run["parameters/d_model_ff"] = d_model_ff
run["parameters/nhead"] = nhead
run["parameters/num_layers_enc"] = num_layers_enc
run["parameters/num_layers_dec"] = num_layers_dec
run["parameters/dropout_rate"] = dropout_rate


torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.benchmark = False


[neptune] [warning] NeptuneUnsupportedType: You're attempting to log a type that is not directly supported by Neptune (<class 'torch.device'>).
        Convert the value to a supported type, such as a string or float, or use stringify_unsupported(obj)
        for dictionaries or collections that contain unsupported values.
        For more, see https://docs-legacy.neptune.ai/help/value_of_unsupported_type


## Helper functions

In [7]:

def tour_length(G, tour):
    """
    Compute the length of a tour. A tour is a list having elments 0 and -1 equal
    """
    n = len(tour) - 1
    assert tour[0] == tour[-1], "Not valid tour"
    estimated = 0
    for i in range(n):
        estimated += G[tour[i]][tour[i + 1]]['weight']
    return estimated

def greedy_algorithm(G):
    """
    Run the value of the greedy approximation algorithm on graph G
    """
    return tour_length(G, greedy_tsp(G, weight='weight'))

def random_tour(G, seed = 42):
    """
    Return the value of a random tour
    """
    np.random.seed(seed)
    n = G.number_of_nodes()
    tour = [0]
    for i in range(1, n):
        next_node = np.random.choice([j for j in range(n) if j not in tour])
        tour.append(next_node)
    tour.append(0)
    return tour_length(G, tour)


def transformer_tsp(G, model, DEVICE = 'cpu'):
    """
    Evaluate your (trained) model on G
    TODO: If you used some masks, add them when needed.
    """
    # Set the model in evaluation mode
    model.eval()

    # Note: number of edges is constant ed equal to n(n-1)/2
    n = G.number_of_nodes()
    E = G.number_of_edges()


    # Get node coordinates
    attr = nx.get_node_attributes(G, 'pos')
    x = []
    for i in range(n):
        x.append(torch.tensor(attr[i], dtype=torch.float32))

    # From list of tensors to tensor
    x = torch.stack(x)

    tour = [0]
    y = torch.tensor(tour, dtype=torch.long)
    x = torch.stack(x)
    x = x.to(DEVICE).unsqueeze(0)
    y = y.to(DEVICE).unsqueeze(0)

    out = model(x, y)

    while len(tour) < n:
        _, idx = torch.topk(out, n, dim=2)
        for i in range(n):
            if idx[0,-1, i] not in tour:
                tour.append(idx[0, -1, i])
                break
        y = torch.tensor(tour)
        y = y.to(DEVICE).unsqueeze(0)
        out = model(x, y)

    tour = [int(i) for i in tour] + [0]
    return tour_length(G, tour)



def gap(G, model = None, model_GA = None, random_seed = 42, device = 'cpu'):
    """
    Compute the gap between the optimal solution on graph G and all the analyzed methods
    """

    # Optimal value (hard-coded in the graph)
    TSP = sum([G[i][j]['weight']*G[i][j]['tour'] for (i, j) in G.edges()]) # Optimal

    # Gaps dictionary
    gaps = {'greedy' : 0, 'random' : 0, 'transformer_tsp': 0, 'transformer_tsp_acc_grad': 0}
    gaps['greedy'] = 100* (greedy_algorithm(G) -  TSP) / TSP
    gaps['random'] = 100 * (random_tour(G, random_seed) - TSP) / TSP
    if model is not None:
        gaps['transformer_tsp'] = 100 * (transformer_tsp(G, model, DEVICE=device) - TSP) / TSP
    else:
        gaps['transformer_tsp'] = float('inf')

    if model_GA is not None:
        gaps['transformer_tsp_acc_grad'] = 100 * (transformer_tsp(G, model_GA, DEVICE=device) - TSP) / TSP
    else:
        gaps['transformer_tsp_acc_grad'] = float('inf')
    return gaps

#read a pkl file
def read_pkl(fname):
    with open(fname, 'rb') as f:
        data = pickle.load(f)
    return data

#for neptune data acquisition
def print_data_before_parentheses(input_str):
  # Prints the data before the first '(' in a string.
  first_parenthesis_index = input_str.find('(')
  if first_parenthesis_index != -1:
    return input_str[:first_parenthesis_index]
  else:
    return input_str

## Dataset & Dataloader

### Task 1.1

In [8]:
if dummy_data:
  fname = '/content/drive/MyDrive/data/dummy_20_DLL_ass4.pkl'
  #Load pickle data file
  data = read_pkl(fname)
  #Get single item
  data_item = data[0]
  print(f"The dataset is of - {type(data)}")
  print(f"The single item in the dataset is of - {type(data_item)}")
else:
  fname = '/content/drive/MyDrive/data/train_20_DLL_ass4.pkl'
  #Load pickle data file
  train_data = read_pkl(fname)
  data_item = train_data[0]
  print(f"The dataset is of - {type(train_data)}")
  print(f"The single item in the dataset is of - {type(data_item)}")

  #load val data
  fname_val = '/content/drive/MyDrive/data/valid_20_DLL_ass4.pkl'
  val_data = read_pkl(fname_val)

The dataset is of - <class 'list'>
The single item in the dataset is of - <class 'tuple'>


###Task 1.2

In [ ]:
print(data_item)

(<networkx.classes.graph.Graph object at 0x7f2f105ffb50>, [0, 3, 14, 2, 9, 6, 19, 13, 12, 16, 7, 18, 8, 17, 5, 11, 10, 15, 1, 4, 0])


In [ ]:
graph = data_item[0]

In [ ]:
#graph node attribute
graph.nodes[0]["pos"]

(0.6049077053425551, 0.5748590937018008)

In [ ]:
#graph edge attribute
for u, v, datax in graph.edges.data():
    if 'tour' in datax and 'weight' in datax:
      if datax['tour'] == 1:
        print(f"Edge: ({u}, {v}), Tour: {datax['tour']}, Weight: {datax['weight']}")
        break

Edge: (0, 3), Tour: 1, Weight: 0.08154537102129383


### Task 1.3

In [9]:
class TSP_Dataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        graph = self.data[idx][0]
        n = graph.number_of_nodes()
        # Create the node coordinates (X)
        # Assuming 'pos' attribute contains coordinates
        node_coords = []
        for i in range(len(self.data[idx][1])-1):
          node = self.data[idx][1][i]
          node_coords.append(graph.nodes[node]['pos'])

        X = torch.tensor(node_coords, dtype=torch.float32)

        # Create tour (y)
        y = torch.tensor(self.data[idx][1], dtype=torch.long)

        return X, y

In [10]:
#apply tsp_dataset
if dummy_data:
  dataset = TSP_Dataset(data)
  print("Shape of X:", dataset[0][0].shape)
  print("Shape of y:", dataset[0][1].shape)
else:
  train_dataset = TSP_Dataset(train_data)
  val_dataset = TSP_Dataset(val_data)
  print("Shape of X:", train_dataset[0][0].shape)
  print("Shape of y:", train_dataset[0][1].shape)

Shape of X: torch.Size([20, 2])
Shape of y: torch.Size([21])


### Task 1.4

In [11]:
if dummy_data:
  train_dataset, val_dataset, test_dataset = random_split(dataset, [0.7, 0.2, 0.1])
  # Create DataLoader objects - Train
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  # val
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
  #test
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

else:
  #train
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  #val
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)



## Model

Positional Embedding

In [12]:
# Corrected PositionalEncoding class to handle batch_first=True inputs
class PositionalEncoding(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1, maxlen: int = 5000):
        super(PositionalEncoding, self).__init__()
        # Denominator for the sinusoidal arguments
        den = torch.exp(-torch.arange(0, emb_size, 2) * math.log(10000) / emb_size)
        # Position tensor (maxlen, 1)
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        # Initialize positional embedding matrix (maxlen, emb_size)
        pos_embedding = torch.zeros((maxlen, emb_size))
        # Apply sine to even indices in the embedding
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        # Apply cosine to odd indices in the embedding
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        # The original unsqueeze(-2) adds a dimension for batch, making it (maxlen, 1, emb_size).
        # For batch_first=True, we want (1, maxlen, emb_size) so it can broadcast across batches.
        # We'll adjust the forward pass to handle this.
        pos_embedding = pos_embedding.unsqueeze(0) # Change to (1, maxlen, emb_size) for broadcasting

        self.dropout = nn.Dropout(dropout)
        # Register as a buffer so it's part of the model's state but not a learnable parameter
        self.register_buffer('pos_embedding', pos_embedding)

    def forward(self, token_embedding):
        # token_embedding shape: (batch_size, sequence_length, emb_size)
        # self.pos_embedding shape: (1, maxlen, emb_size)
        # We need to slice pos_embedding based on the sequence length of the input
        # token_embedding.size(1) gives the sequence length when batch_first=True
        return self.dropout(token_embedding + self.pos_embedding[:, :token_embedding.size(1), :])

Transformer network

In [13]:
# Custom transformer model for TSP
class TSP_Transformer(nn.Module):
    def __init__(self, num_cities, d_model_enc, d_model_dec, d_model_ff, nhead, num_layers_enc, num_layers_dec, dropout_rate=0.4):
        super(TSP_Transformer, self).__init__()
        self.num_cities = num_cities # Store num_cities for the output layer

        # Encoder: Embed 2D Cartesian coordinates (x, y) into d_model_enc dimension
        self.cartesian_embedding = nn.Linear(2, d_model_enc)

        # Decoder: Embed city indices into d_model_dec dimension
        # num_cities should be the total number of cities (vocabulary size)
        self.city_embedding = nn.Embedding(num_cities, d_model_dec)

        # Decoder: Positional encoding for the sequence of cities
        self.positional_encoding_dec = PositionalEncoding(d_model_dec, dropout=dropout_rate, maxlen=num_cities) # Maxlen can be num_cities or higher

        # Transformer Encoder layers
        # batch_first=True means input/output tensors are (batch_size, sequence_length, features)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model_enc, nhead, dim_feedforward=d_model_ff, dropout=dropout_rate, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers_enc)

        # Linear projection to match encoder output dimension to decoder input dimension
        # This is used for the cross-attention mechanism in the decoder
        self.encode_to_decoder_prj = nn.Linear(d_model_enc, d_model_dec)

        # Transformer Decoder layers
        decoder_layers = nn.TransformerDecoderLayer(
            d_model_dec, nhead, dim_feedforward=d_model_ff, dropout=dropout_rate, batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layers, num_layers_dec)

        # Generator layer: Projects decoder output to logits for each city
        # This layer should output a score for each of the 'num_cities' possible cities
        self.output_layer = nn.Linear(d_model_dec, num_cities) # Corrected output dimension

    def generate_square_subsequent_mask(self, sz):
        # Generates a causal mask for the decoder to prevent attending to future tokens
        # The mask ensures that position i can only attend to positions j <= i.
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        # Fill lower triangle with -inf (for masked attention) and upper with 0.0 (for unmasked)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, coords, initial_order):
        # coords: (batch_size, num_cities_in_problem, 2) - Input city coordinates
        # initial_order: (batch_size, current_tour_length) - Partially built tour (city indices)

        # 1. Embed city coordinates for the encoder
        # Output shape: (batch_size, num_cities_in_problem, d_model_enc)
        cartesian_embeddings = self.cartesian_embedding(coords)
        # print(f"\n3.cartesian_embeddings: {cartesian_embeddings.shape}") # Debug print

        # 2. Pass embedded coordinates through the Transformer Encoder
        # encoder_output shape: (batch_size, num_cities_in_problem, d_model_enc)
        encoder_output = self.transformer_encoder(cartesian_embeddings)
        # print(f"\n4.encoder_output: {encoder_output.shape}") # Debug print

        # 3. Project encoder output to the decoder's dimension
        # This output serves as 'memory' for the decoder's cross-attention.
        # Output shape: (batch_size, num_cities_in_problem, d_model_dec)
        encoder_output = self.encode_to_decoder_prj(encoder_output)
        # print(f"\n5.encoder_output: {encoder_output.shape}") # Debug print

        # 4. Embed the initial (partial) tour sequence for the decoder
        # initial_order contains city indices.
        # Output shape: (batch_size, current_tour_length, d_model_dec)
        city_embeddings = self.city_embedding(initial_order)
        # print(f"\n6.city_embeddings: {city_embeddings.shape}") # Debug print

        # 5. Add positional encoding to the decoder embeddings
        # This provides information about the order of cities in the partial tour.
        # Output shape: (batch_size, current_tour_length, d_model_dec)
        city_embeddings = self.positional_encoding_dec(city_embeddings)
        # print(f"\n7.city_embeddings: {city_embeddings.shape}") # Debug print

        # 6. Create a causal mask for the decoder's self-attention
        # This ensures the decoder only attends to previously predicted cities.
        seq_len = initial_order.size(1) # current_tour_length
        mask = self.generate_square_subsequent_mask(seq_len).to(coords.device) # Move mask to the correct device
        # print(f"Mask shape: {mask.shape}") # Debug print

        # 7. Pass through the Transformer Decoder
        # tgt: The target sequence (partial tour embeddings with positional encoding)
        # memory: The output from the encoder (contextualized city embeddings)
        # tgt_mask: The causal mask for self-attention in the decoder
        # Output shape: (batch_size, current_tour_length, d_model_dec)
        decoder_output = self.transformer_decoder(city_embeddings, encoder_output, tgt_mask=mask)
        # print(f"\n8.decoder_output: {decoder_output.shape}") # Debug print

        # 8. Final output layer to generate logits for the next city
        # For each position in the partial tour, predict the next city's logits.
        # Output shape: (batch_size, current_tour_length, num_cities)
        output = self.output_layer(decoder_output)
        # print(f"\n9.output: {output.shape}") # Debug print
        # print("\n-------------")

        # The output `output` now contains logits for each possible city at each step
        # in the sequence. For TSP, you typically want the logits for the *last*
        # position in the sequence to predict the *next* city.
        # You might want to return output[:, -1, :] for the logits of the next city,
        # or the full sequence of logits if you're using a loss that expects it.
        # For a typical sequence-to-sequence setup, you'd apply CrossEntropyLoss
        # on output.view(-1, num_cities) and target_sequence.view(-1).
        return output


## Training

In [14]:
# Define a Transformer model by using the hyper parameters
model = TSP_Transformer(num_cities=num_cities, d_model_enc=d_model_enc, d_model_dec=d_model_dec, d_model_ff=d_model_ff, nhead=nhead, num_layers_enc=num_layers_enc, num_layers_dec=num_layers_dec, dropout_rate=dropout_rate)
#upload to neptune
run["sys/tags"].add(print_data_before_parentheses(str(model)))

In [15]:
print(f"The shape of the first batch of data is: {next(iter(train_loader))[0].shape}")
print(f"The shape of the first batch of labels is: {next(iter(train_loader))[1].shape}")

The shape of the first batch of data is: torch.Size([32, 20, 2])
The shape of the first batch of labels is: torch.Size([32, 21])


Set the loss fn and optimizer

In [16]:
criterion = nn.CrossEntropyLoss()
run["parameters/loss_criterion"] = print_data_before_parentheses(str(criterion))
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-3)
run["parameters/optimizer"] = print_data_before_parentheses(str(optimizer))


### Training WITHOUT gradient accumulation

In [17]:
train_losses = []
val_losses = []
#training Structure inspired from Exercise 5
#Validate model loss
def validate_model(model, loader, criterion, device):

    #set model to eval
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_decoder_input = y[:, :-1]
            y_decoder_output = y[:, 1:]

            output = model(X, y_decoder_input)
            loss = criterion(output.view(-1, output.size(-1)),y_decoder_output.type(torch.float32))
            val_loss += loss.item()

    avgLoss = val_loss / len(loader)
    return avgLoss
#train_wo_accu
def train_without_accumulation(model, train_loader, val_loader, criterion, optimizer, device, epochs=10):
    #shift model to gpu
    model = model.to(device)
    for epoch in range(epochs):
      #set model to train
        model.train()
        train_loss = 0
        train_loss_avg = 0
        for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Training . Batch Ind Loss {train_loss_avg}", unit="batch"):
            X, y = X.to(device), y.to(device)
            #y shifted approperly as per model instructions
            y_decoder_input = y[:, :-1]

            y_decoder_output = y[:, 1:]

            # Forward pass
            optimizer.zero_grad()

            output = model(X, y_decoder_input)

            # Compute loss
            loss = criterion(output.view(-1, output.size(-1)),y_decoder_output.type(torch.float32))
            loss.backward()
            optimizer.step()

            run["train/Batch_loss"].append(loss.item())
            train_loss += loss.item()
            # print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

        #validation data evaluation
        train_loss_avg = train_loss / len(train_loader)
        val_loss = validate_model(model, val_loader, criterion, device)

        run["train/loss"].append(train_loss_avg)
        run["val/loss"].append(val_loss)

        #store epochs in runs
        run["train/epoch"].append(epoch+1)

        print(f"Epoch {epoch + 1}, Train Loss: {train_loss_avg:.4f}, Validation Loss: {val_loss:.4f}")

        train_losses.append(train_loss / len(train_loader))
        val_losses.append(val_loss)
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss without accumlation')
    plt.legend()
    plt.savefig('train_val_loss_wo_accu.png')

    return train_losses, val_losses

print("Training without gradient accumulation:")
tl, vl= train_without_accumulation(model, train_loader, val_loader, criterion, optimizer, device, epochs)

Training without gradient accumulation:


Epoch 1/5 Training . Batch Ind Loss 0:   0%|          | 0/3 [00:00<?, ?batch/s]


ValueError: Expected input batch_size (640) to match target batch_size (32).

In [ ]:
#upload to neptune
run["validation/Train_Val_wo_accu"].upload("train_val_loss_wo_accu.png")

In [ ]:
# Clear the current figure
plt.clf()

### Training WITH gradient accumulation

In [ ]:
#Gradient accumlation training

#I have refrenced the code from an article in wandb

def train_with_accumulation(model, train_loader, val_loader, criterion, optimizer, device, epochs=10, accumulation_steps=2):
    model = model.to(device)
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()  # Initialize gradients
        for i, (X, y) in tqdm(enumerate(train_loader), desc=f"Epoch {epoch+1}/{epochs} Training Accumlation Grad", unit="batch"):
            X, y = X.to(device), y.to(device)

            y_decoder_input = y[:, :-1]
            y_decoder_output = y[:, 1:]

            output = model(X, y_decoder_input)
            loss = criterion(output.view(-1, output.size(-1)),y_decoder_output.type(torch.float32))
            # Normalizing the loss
            loss = loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:  # Gradient accumulation
                optimizer.step()
                optimizer.zero_grad()

            train_loss += loss.item()*accumulation_steps

        train_loss_avg = (train_loss / len(train_loader))
        val_loss = validate_model(model, val_loader, criterion, device)


        run["train_GA/loss"].append(train_loss_avg)
        run["val_GA/loss"].append(val_loss)

        run["train_GA/epoch"].append(epoch+1)

        print(f"Epoch {epoch + 1}, Train Loss: {train_loss_avg:.4f}, Validation Loss: {val_loss:.4f}")

        train_losses.append(train_loss_avg)
        val_losses.append(val_loss)

    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss with Gradient Accumulation')
    plt.legend()
    plt.savefig('train_val_loss_w_accu.png')

    return train_losses, val_losses

In [ ]:
print("Training with gradient accumulation:")

model_GA = TSP_Transformer(num_cities=num_cities, d_model_enc=d_model_enc, d_model_dec=d_model_dec, d_model_ff=d_model_ff, nhead=nhead, num_layers_enc=num_layers_enc, num_layers_dec=num_layers_dec, dropout_rate=dropout_rate)

optimizer_GA = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-3)

tl_ga, vl_ga = train_with_accumulation(model_GA, train_loader, val_loader, criterion, optimizer_GA, device, epochs, accumulation_steps=2)

In [ ]:
#save plt image as train_val_loss_wo_accu

#upload to neptune
run["validation/Train_Val_w_accu"].upload("train_val_loss_w_accu.png")

In [ ]:
# Clear the current figure
plt.clf()

## Testing

In [ ]:
if dummy_data:
  #create a 10 size subset of initial data read in pkl format
  testdata = data[:10]
else:
  #read the pkl file
  testdata = read_pkl("/content/drive/MyDrive/data/test_20_DLL_ass4.pkl")


In [ ]:
greddy_tour_list = []
random_tour_list = []
transformer_tour_list = []
transformer_accu_tour_list = []
for i in range(len(testdata)):
  greedy_tour_val = greedy_algorithm(testdata[i][0])
  random_tour_val = random_tour(testdata[i][0])
  transformer_tour_val = transformer_tsp(testdata[i][0], model, DEVICE=device)
  transformer_accu_tour_val = transformer_tsp(testdata[i][0], model_GA, DEVICE=device)
  #print with labels of the algorithm and save them in the list
  #print(f"Greedy Algorithm Tour: {greedy_tour_val}")
  #print(f"Random Algorithm Tour: {random_tour_val}")
  #print(f"Transformer Algorithm Tour: {transformer_tour_val}")
  #print(f"Transformer Algorithm Tour with Gradient Accumulation: {transformer_accu_tour_val}")

  greddy_tour_list.append(greedy_tour_val)
  random_tour_list.append(random_tour_val)
  transformer_tour_list.append(transformer_tour_val)
  transformer_accu_tour_list.append(transformer_accu_tour_val)

  #save list in neptune run
  run["test/greedy_tour"].append(greedy_tour_val)
  run["test/random_tour"].append(random_tour_val)
  run["test/transformer_tour"].append(transformer_tour_val)


In [ ]:

skip_testdata= 3 # cut short test data if too large
gaps_list = []
for i in range(len(testdata)):
  if i%skip_testdata==0:

    gaps = gap(testdata[i][0], model, model, device=device)
    gaps_list.append(gaps)


gaps_pd= pd.DataFrame(gaps_list)
#plot a box plot

gaps_pd.plot(kind='box')

#add title
plt.title("Gap Distribution")
#xlabel
plt.xlabel("Algorithms")
#ylabel
plt.ylabel("Gap")
#show
plt.savefig('Gap_Distribution.png')

In [ ]:

#upload to neptune
run["validation/Gap Distribution"].upload("Gap_Distribution.png")

In [ ]:
run.stop()